# RealThor v0.1 — Listing Supply Intelligence

This notebook is the starting point for Module 1. It shows RealThor running directly in a notebook before any project structure or versioning is in place.

**Use case:** Given demand and supply signals for a region, should we prioritize new listings there?

> **Note:** This notebook intentionally has the prompt inline (not externalized). The demo shows why that is a problem.

## Setup

In [1]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
provider = os.getenv("LLM_PROVIDER", "openai")
if provider == "vocareum":
    client = OpenAI(base_url="https://openai.vocareum.com/v1", api_key=os.getenv("VOCAREUM_API_KEY"))
else:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Load Ciudaty Data

In [4]:
demand = pd.read_csv("demand_signals.csv")

In [5]:
print("Demand signals:")
display(demand)

Demand signals:


,region,property_type,bedrooms,search_volume,lead_count,recent_sales,demand_score
0,west,apartment,2,1240,86,34,0.91
1,north,apartment,1,620,31,12,0.54
2,south,house,3,710,42,18,0.68
3,east,apartment,2,980,73,21,0.82
4,center,studio,1,430,19,8,0.47


In [6]:
supply = pd.read_csv("supply_snapshot.csv")

In [7]:
print("\nSupply snapshot:")
display(supply)


Supply snapshot:


,region,property_type,bedrooms,active_listings,avg_days_on_market,supply_score
0,west,apartment,2,18,21,0.22
1,north,apartment,1,77,54,0.81
2,south,house,3,36,39,0.55
3,east,apartment,2,29,28,0.37
4,center,studio,1,91,63,0.88


## RealThor v0.1

The system prompt is defined inline here. In Video 3, we will extract it to a versioned file.

In [8]:
# NOTE: prompt is inline — this is the problem we will fix soon
SYSTEM_PROMPT = """
You are RealThor, a Listing Supply Intelligence agent for a PropTech operating in Ciudaty.

Your task is to analyze demand and supply signals for a given region and property profile.

Use the available data to recommend whether the company should prioritize bringing more
properties of that type to the platform.

Your recommendation must include:
- priority: high, medium, or low
- reasoning based on demand
- reasoning based on supply
- recommended action
- confidence: high, medium, or low

Do not make claims that are not supported by the provided data.
"""

In [9]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]

    if d.empty or s.empty:
        return "No data available for the requested segment."

    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [10]:
def ask_realthor(region: str, property_type: str, bedrooms: int) -> str:
    context = build_context(region, property_type, bedrooms)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ],
    )
    return response.choices[0].message.content

## First Recommendation

In [11]:
recommendation = ask_realthor(region="west", property_type="apartment", bedrooms=2)
print(recommendation)

- **Priority:** High
- **Reasoning based on demand:** The demand for 2-bedroom apartments in the west region is strong, as indicated by a high search volume of 1240 and a lead count of 86. The demand score of 0.91 suggests that there is a significant interest and active engagement from potential buyers or renters.
- **Reasoning based on supply:** The supply of 2-bedroom apartments in the west region is currently low, with only 18 active listings and an average of 21 days on market. The supply score of 0.22 indicates that there is a considerable gap between demand and available properties, showcasing a limited offering.
- **Recommended action:** The company should prioritize bringing more 2-bedroom apartments to the platform in the west region to capture the high demand and rectify the low supply situation.
- **Confidence:** High


## Try Another Segment

In [12]:
recommendation = ask_realthor(region="north", property_type="apartment", bedrooms=1)
print(recommendation)

**Priority:** Medium

**Reasoning based on Demand:**
The demand signals for 1-bedroom apartments in the north region indicate a search volume of 620, with a lead count of 31 and recent sales totaling 12. The demand score of 0.54 suggests a moderate level of interest in this property type, indicating that there are interested buyers but the demand is not exceptionally high.

**Reasoning based on Supply:**
On the supply side, there are currently 77 active listings for 1-bedroom apartments in the north, with an average of 54 days on the market. The supply score of 0.81 suggests a relatively high level of supply compared to demand. This indicates that while there are some sales happening (12 recently), the existing listings are not being absorbed quickly, which can imply that the current market is somewhat saturated.

**Recommended Action:**
While there is some demand for 1-bedroom apartments, the supply is relatively high, suggesting that the property profile may not be the highest priori

---

## What we just built

RealThor works. But:

- The system prompt is **hardcoded inline**. If we change it, we lose the previous version.
- There is **no project structure** — the notebook and the data are in the same flat folder.
- There is **no environment pinning** — `pip install openai` could install any version.
- If a teammate runs this notebook tomorrow and gets a different answer, **we cannot explain why**.

In the next videos, we fix all of this.